# Session 2: Kirchhoff's two laws
**A. Real-world problem.** A microcontroller board runs on 3.3 V but your battery pack gives
5 V. Two resistors can drop the voltage, but when the sensor you connect draws current the
output sags. How do we predict the sag and design around it? And what if two batteries feed
the same circuit: which way does the current flow?

**Plan.** (1) state the current law (KCL) and voltage law (KVL), (2) dividers, (3) the loading
effect, (4) a two-loop circuit solved with simultaneous equations, (5) a design inverse problem.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the two laws
**KCL (current law):** at any node the currents flowing in add up to the currents flowing out
(charge does not pile up). **KVL (voltage law):** going once around any closed loop, the
voltage rises and drops add up to zero (energy is conserved).

## Step 2: the voltage divider
Two resistors in series across V_in: the same current I = V_in / (R1 + R2) passes both, so
V_out across R2 is V_in * R2 / (R1 + R2). Current divider: a total current I_t splitting into
two parallel branches gives I1 = I_t * R2 / (R1 + R2) (the smaller resistor takes more).

In [ ]:
def divider_out(vin, r1, r2):
    """Unloaded voltage divider: V_out across R2 (volts)."""
    if r1 < 0 or r2 <= 0:
        raise ValueError("R1 must be >= 0 and R2 > 0")
    return vin * r2 / (r1 + r2)


def current_divider(i_total, r1, r2):
    """Split a total current between two parallel resistors. Returns (i1, i2) in amperes."""
    if r1 <= 0 or r2 <= 0:
        raise ValueError("both resistors must be positive")
    return i_total * r2 / (r1 + r2), i_total * r1 / (r1 + r2)


def parallel(a, b):
    """Two resistors in parallel (ohm)."""
    return a * b / (a + b)

## Step 3: the loading effect
A load R_L connected across the output sits in parallel with R2. The lower resistor becomes
R2 || R_L = R2 R_L / (R2 + R_L), which is smaller, so the output drops.
Check by hand: 5 V, R1 = 1700, R2 = 3300 ohm gives 3.30 V unloaded. With R_L = 4700 ohm the
lower part becomes 3300 || 4700 = 1938.75 ohm and V_out = 5 * 1938.75 / (1700 + 1938.75) = 2.664 V.

In [ ]:
def loaded_out(vin, r1, r2, rl=np.inf):
    """Divider output when a load rl (ohm) is connected across R2. rl = inf means no load."""
    lower = r2 if np.isinf(rl) else parallel(r2, rl)
    return divider_out(vin, r1, lower)


def supply_current(vin, r1, r2, rl=np.inf):
    """Current drawn from the source (A)."""
    lower = r2 if np.isinf(rl) else parallel(r2, rl)
    return vin / (r1 + lower)

## Step 4: a circuit with two batteries (two loops)
Battery V1 with R1 and battery V2 with R2 both feed node A; R3 goes from A to ground (0 V).
Let I1 and I2 flow out of the batteries towards A, so R3 carries I1 + I2. KVL on loop 1 and
loop 2:
    V1 = R1 I1 + R3 (I1 + I2)        ->  (R1 + R3) I1 +  R3 I2       = V1
    V2 = R2 I2 + R3 (I1 + I2)        ->   R3 I1       + (R2 + R3) I2 = V2
Two equations, two unknowns. By Cramer's rule: det = (R1+R3)(R2+R3) - R3^2,
I1 = (V1 (R2+R3) - R3 V2) / det and I2 = ((R1+R3) V2 - R3 V1) / det.
A negative current simply means the current really flows the other way.

In [ ]:
def two_loop_by_hand(v1, v2, r1, r2, r3):
    """Closed-form (Cramer) solution. Returns (i1, i2, i3, v_a) with i3 = i1 + i2 the R3 current."""
    det = (r1 + r3) * (r2 + r3) - r3 * r3
    i1 = (v1 * (r2 + r3) - r3 * v2) / det
    i2 = ((r1 + r3) * v2 - r3 * v1) / det
    return i1, i2, i1 + i2, r3 * (i1 + i2)


def two_loop(v1, v2, r1, r2, r3):
    """The same circuit solved by numpy: A x = b with x = (I1, I2)."""
    a = np.array([[r1 + r3, r3],
                  [r3, r2 + r3]], dtype=float)
    b = np.array([v1, v2], dtype=float)
    i1, i2 = np.linalg.solve(a, b)
    return float(i1), float(i2), float(i1 + i2), float(r3 * (i1 + i2))


def kcl_residual(i1, i2, i3):
    """Currents into node A minus current out through R3 (should be 0)."""
    return i1 + i2 - i3


def kvl_residual(v1, r1, i1, v_a):
    """Loop 1 around V1, R1 and R3: rise V1 minus drop R1*I1 minus node voltage (should be 0)."""
    return v1 - r1 * i1 - v_a

## Step 5: the design (inverse) problem
We want exactly V_target at the output WITH the load connected. Choose R1, then the loaded
lower part must be R_low = R1 V_target / (V_in - V_target), and since R_low = R2 || R_L,
1/R2 = 1/R_low - 1/R_L (needs R_low < R_L).

In [ ]:
def design_r2(vin, vtarget, r1, rl):
    """R2 that gives vtarget across the load rl, given R1. Raises ValueError if impossible."""
    if not 0 < vtarget < vin:
        raise ValueError("target must lie between 0 and the input voltage")
    r_low = r1 * vtarget / (vin - vtarget)
    if r_low >= rl:
        raise ValueError("load is too heavy: even with R2 open the load alone pulls the output too low")
    return 1.0 / (1.0 / r_low - 1.0 / rl)


def stiffness_table(vin=5.0, vtarget=3.3, rl=4700.0, r1_list=(100, 220, 470, 1000, 2200)):
    """For each R1: the R2 that hits the target with the load, the supply current and the
    error if the load is removed (an unloaded divider with the same resistors)."""
    rows = []
    for r1 in r1_list:
        r2 = design_r2(vin, vtarget, r1, rl)
        rows.append((r1, r2, supply_current(vin, r1, r2, rl), divider_out(vin, r1, r2) - vtarget))
    return rows


if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print("loading effect, 5 V, R1 = 1700, R2 = 3300 ohm")
    for rl in (np.inf, 47000, 10000, 4700, 1000):
        print(f"  load {rl:>8} ohm -> V_out = {loaded_out(5.0, 1700, 3300, rl):.3f} V")
    print("two-loop (10 V, 4 V, 100, 200, 300 ohm): I1, I2, I3, V_A =", np.round(two_loop(10, 4, 100, 200, 300), 5))
    print("design with R1 = 1000 ohm, load 4700 ohm: R2 =", round(design_r2(5, 3.3, 1000, 4700), 1), "ohm")
    for r1, r2, i, e in stiffness_table():
        print(f"  R1 {r1:5d}  R2 {r2:8.1f}  supply {1000 * i:6.2f} mA  error without load {e:+.3f} V")

    rl = np.linspace(200, 20000, 400)
    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(rl, [loaded_out(5, 1700, 3300, x) for x in rl], label="R1 = 1700, R2 = 3300 ohm")
    ax.plot(rl, [loaded_out(5, 1000, 3300, x) for x in rl], label="R1 = 1000, R2 = 3300 ohm")
    ax.axhline(3.3, color="gray", ls=":")
    ax.set_xlabel("load resistance (ohm)")
    ax.set_ylabel("output voltage (V)")
    ax.set_title("A divider sags when the load is light on resistance")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "divider_loading.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    v2 = np.linspace(-4, 12, 81)
    ax2.plot(v2, [1000 * two_loop(10, x, 100, 200, 300)[0] for x in v2], label="I1 (battery 1)")
    ax2.plot(v2, [1000 * two_loop(10, x, 100, 200, 300)[1] for x in v2], label="I2 (battery 2)")
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel("battery 2 voltage V2 (V)")
    ax2.set_ylabel("battery current (mA)")
    ax2.set_title("Two batteries on one node (V1 = 10 V): I2 changes sign")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "two_loop_currents.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Call `loaded_out(5, 1700, 3300, rl)` for rl = 1000, 4700, 47000 and 470000 ohm and describe what happens to the error as the load becomes lighter.
2. **Stretch.** Change the batteries of `two_loop` to V1 = 10 V and V2 = 6.545 V (equal to V_A) and predict I2 before running it. Then find the V2 where I2 changes sign for the same resistors.
3. **Explain.** Explain in your own words why the current divider gives the smaller resistor the larger current, using the fact that both resistors have the same voltage across them.

## Assessment
- **A. Conceptual.** Why does connecting a load always lower the output of a voltage divider, and when can you safely ignore the load?
- **B. Numerical.** A 9 V source feeds R1 = 2 kohm in series with R2 = 1 kohm. Find the unloaded output, then the output with a 1 kohm load across R2. Check both with KVL.
- **C. Graph and simulation interpretation.** In the loading graph for R1 = 1700 ohm and R2 = 3300 ohm, at roughly what load resistance does the output fall to 3.0 V (about 9 percent below the target)? Which direction does the curve move if R2 is made smaller?
- **D. Design and optimisation challenge.** Design: the output must be exactly 3.3 V from 5 V with a 4.7 kohm load. Compare R1 = 220 ohm, 470 ohm and 1000 ohm. For each, find R2, the source current and the error if the load is later unplugged. Which do you choose and why?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    i1, i2, i3, va = L.two_loop(10, 4, 100, 200, 300)
    checks.append(("worked example: two-loop circuit gives I1 = 34.55 mA, I2 = -12.73 mA, V_A = 6.545 V",
                   abs(i1 - 0.034545) < 1e-6 and abs(i2 + 0.012727) < 1e-6 and abs(va - 6.54545) < 1e-4))
    h = L.two_loop_by_hand(10, 4, 100, 200, 300)
    checks.append(("hand (Cramer) solution agrees with numpy.linalg.solve", np.allclose(h, (i1, i2, i3, va), atol=1e-12)))
    checks.append(("KCL at node A: currents in minus current out is zero", abs(L.kcl_residual(i1, i2, i3)) < 1e-12))
    checks.append(("KVL round loop 1 (V1, R1, R3) sums to zero", abs(L.kvl_residual(10, 100, i1, va)) < 1e-12))
    checks.append(("KVL round loop 2 (V2, R2, R3) sums to zero", abs(L.kvl_residual(4, 200, i2, va)) < 1e-12))
    checks.append(("power balance: sources supply what the resistors dissipate",
                   abs(10 * i1 + 4 * i2 - (100 * i1 ** 2 + 200 * i2 ** 2 + 300 * i3 ** 2)) < 1e-12))
    checks.append(("hand calculation: 5 V, 1700 and 3300 ohm divider gives 3.300 V unloaded",
                   abs(L.divider_out(5, 1700, 3300) - 3.3) < 1e-9))
    checks.append(("loading effect: a 4700 ohm load pulls the same divider down to 2.664 V",
                   abs(L.loaded_out(5, 1700, 3300, 4700) - 2.664) < 5e-4))
    a, b = L.current_divider(0.03, 100, 200)
    checks.append(("current divider: 30 mA into 100 ohm and 200 ohm splits 20 mA and 10 mA",
                   abs(a - 0.02) < 1e-12 and abs(b - 0.01) < 1e-12 and abs(a + b - 0.03) < 1e-12))
    checks.append(("a very large load changes almost nothing", abs(L.loaded_out(5, 1700, 3300, 1e9) - 3.3) < 1e-4))
    r2 = L.design_r2(5.0, 3.3, 1000, 4700)
    checks.append(("design: R1 = 1000 and a 4700 ohm load need R2 = 3307 ohm for exactly 3.300 V",
                   abs(r2 - 3307.0) < 0.5 and abs(L.loaded_out(5, 1000, r2, 4700) - 3.3) < 1e-9))
    checks.append(("standard value 3300 ohm lands within 2 percent of 3.3 V and draws under 6 mA",
                   abs(L.loaded_out(5, 1000, 3300, 4700) - 3.3) < 0.066 and L.supply_current(5, 1000, 3300, 4700) < 0.006))
    try:
        L.design_r2(5.0, 3.3, 4700, 4700)
        ok = False
    except ValueError:
        ok = True
    checks.append(("impossible design (R1 = 4700 with a 4700 ohm load) is rejected", ok))
    checks.append(("edge: R1 = 0 passes the full input voltage", abs(L.divider_out(5, 0, 1000) - 5.0) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
